# DocChat — GPU LLM backend on Google Colab

Runs the app's models via Ollama on a free Colab GPU: `qwen2.5:14b` by default (a T4 has room for it and it reads
tables noticeably better than the 7B used on CPU), plus `nomic-embed-text`. Set `MODEL = "qwen2.5:7b"` to match the laptop setup exactly.
and exposes them through a temporary HTTPS tunnel. The DocChat app itself still runs on your laptop; only the
text-to-text LLM calls go here. Documents are still parsed locally — the LLM only ever receives text.

**Steps:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Copy the three `.env` lines printed
by the last cell into `docchat/.env` and restart DocChat.

> ⚠️ The tunnel URL is public while this notebook runs (anyone with the URL can use the model). Stop the runtime after the demo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Install Ollama (official install script) and the tools it needs
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_ORIGINS="*")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); print("Ollama is up"); break
    except Exception:
        time.sleep(1)

In [ ]:
MODEL = "qwen2.5:14b"  # or "qwen2.5:7b"
!ollama pull {MODEL}
!ollama pull nomic-embed-text
!ollama list  # qwen2.5:14b (about 9 GB) must be listed; if not, run this cell again

In [ ]:
# Warm up with the same 10K context the app enforces, and measure speed
r = requests.post("http://localhost:11434/api/chat", json={
    "model": MODEL, "stream": False,
    "messages": [{"role": "user", "content": "Explain retrieval-augmented generation in 3 bullet points. " * 40}],
    "options": {"num_ctx": 10000, "num_predict": 120}}, timeout=600).json()
if "error" in r:  # e.g. the model download in the cell above did not finish
    raise RuntimeError(f"Ollama: {r['error']} -> re-run the pull cell above, then this one")
if r.get("prompt_eval_duration") and r.get("eval_duration"):
    print(f"prompt: {r['prompt_eval_count']} tok @ {r['prompt_eval_count']/(r['prompt_eval_duration']/1e9):.0f} tok/s | "
          f"generation: {r['eval_count']} tok @ {r['eval_count']/(r['eval_duration']/1e9):.1f} tok/s")
else:  # a cached prompt reports no timing
    print("model answered:", r["message"]["content"][:80], "...")

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel, no account needed)
import re
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
assert url, "Tunnel did not start; see tunnel.log"
time.sleep(5)
print("Tunnel check:", requests.get(url + "/api/tags", timeout=30).status_code)
print()
print("Paste these lines into docchat/.env, then restart DocChat:")
print()
print(f"LLM_MODEL={MODEL}")
print(f"LLM_BASE_URL={url}")
print(f"EMBED_BASE_URL={url}")

In [ ]:
# Keep this cell running during the demo (shows that the backend is alive).
while True:
    alive = ollama.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "backend alive" if alive else "STOPPED - rerun the cells above")
    time.sleep(300)